**[🏠 Course Home](../README.md)** | **📖 [Conceptual](../conceptual/09_mcmc_mechanics_from_scratch.ipynb)** | **🤖 [AI-Native](09_mcmc_mechanics_from_scratch.ipynb)** | **🐍 [Python](../python/09_mcmc_mechanics_from_scratch.ipynb)** | **🔵 [R](../r/09_mcmc_mechanics_from_scratch.ipynb)** | **⬅️ [Prev: Ch 08](08_markov_chains_and_detailed_balance.ipynb)** | **➡️ [Next: Ch 10](10_mcmc_production_diagnostics.ipynb)**

*🧭 Part II: Numerical & Sampling Engines — Chapter 09 of 22*

---

# Chapter 9: The Metropolis Protocol — MCMC Mechanics from Scratch
### *Floating-Point Underflow, Log-Scale Decisions, and Step Tuning*

> **🧭 Course Context & Why This Module Matters in the AI Era:**
> In Chapter 8, we discovered that detailed balance allows a Markov chain to
> sample from an unnormalized discrete probability distribution. In this
> chapter, we generalize this principle to continuous parameters via the
> **Metropolis Algorithm**. The Metropolis algorithm is celebrated because it
> completely bypasses the intractable marginal likelihood denominator
> $P(\mathcal{D})$: the evidence constant cancels out in the acceptance ratio.
> However, when software engineers prompt an LLM to generate an MCMC sampler
> from scratch, the AI almost invariably writes likelihood evaluations in raw
> probability space. For realistic datasets ($N > 200$), multiplying raw
> probabilities causes immediate floating-point underflow to zero, producing
> `NaN` crashes or frozen walkers. This chapter establishes the robust,
> log-scale computational contract required for production-grade MCMC.

## Part 1: The Naive Prompt Trap & Forensic Post-Mortem

When developers ask an LLM to build a bespoke Markov Chain Monte Carlo sampler,
they usually describe the algorithm in textbook mathematical terms.

### The Naive Prompt
```text
Write a clean Python function implementing the Metropolis algorithm from
scratch to estimate the mean mu from a dataset of 300 response times. Use a
Normal(170, 15) prior and Normal(mu, 8.0) likelihood. Propose candidate steps
using a normal random walk.
```

### The Plausible-Sounding AI Hallucination
The LLM generates clean-looking, textbook-faithful Python code:
```python
import numpy as np
import scipy.stats as stats


def metropolis_sampler(data, n_samples=5000, start_mu=160.0, step_size=1.0):
  samples = [start_mu]
  current_mu = start_mu

  for _ in range(n_samples):
    # Propose new candidate
    prop_mu = current_mu + np.random.normal(0, step_size)

    # Compute raw prior and likelihood product
    p_curr_prior = stats.norm.pdf(current_mu, 170.0, 15.0)
    p_curr_lik = np.prod(stats.norm.pdf(data, current_mu, 8.0))
    p_curr = p_curr_prior * p_curr_lik

    p_prop_prior = stats.norm.pdf(prop_mu, 170.0, 15.0)
    p_prop_lik = np.prod(stats.norm.pdf(data, prop_mu, 8.0))
    p_prop = p_prop_prior * p_prop_lik

    # Acceptance probability
    alpha = min(1.0, p_prop / p_curr)

    if np.random.uniform(0, 1) < alpha:
      current_mu = prop_mu
    samples.append(current_mu)

  return np.array(samples)
```

```
                 THE FLOATING-POINT UNDERFLOW CATASTROPHE
                 
  Raw Likelihood Product (N = 300 data points):
  
    p(D | mu) = p(y_1 | mu) * p(y_2 | mu) * ... * p(y_300 | mu)
              ~ (0.05) * (0.05) * ... * (0.05)
              ~ (0.05)^300
              ~ 10^(-390)
              
  IEEE 754 Double Precision Subnormal Limit: ~ 10^(-324)
  
              +--------------------------------------------------+
              | Result: p_curr = 0.0,  p_prop = 0.0              |
              | Acceptance Ratio: alpha = 0.0 / 0.0 = NaN!       |
              | Sampler Fails: Walker Freezes or Crashes!        |
              +--------------------------------------------------+
```

---

### Forensic Post-Mortem: Why the AI Failed
The AI's code is mathematically textbook-correct on paper, but computationally
fatal in finite-precision hardware:

1. **Catastrophic IEEE 754 Underflow:**
   Standard 64-bit IEEE 754 floating-point numbers support a minimum positive
   subnormal number of $\approx 4.9 \times 10^{-324}$ and normalized minimum of
   $\approx 2.2 \times 10^{-308}$. For $N=300$ observations, each Gaussian
   density evaluation is roughly $\mathcal{O}(10^{-2})$ to $\mathcal{O}(0.1)$.
   Multiplying 300 probabilities yields an underflow to exact zero:
   $$\prod_{i=1}^{300} p(y_i \mid \mu) \to 0.0000000000000000$$
   Both `p_curr` and `p_prop` collapse to `0.0`. Their ratio `p_prop / p_curr`
   evaluates to `NaN`. In Python, comparisons `np.random.uniform(0, 1) < np.nan`
   always evaluate to `False`. The sampler silently freezes at `start_mu`
   forever!

2. **Log-Scale Arithmetic is Mandatory:**
   A calibrated probabilistic engineer works exclusively in **log-posterior**
   space. The product of probabilities transforms into a stable sum of
   log-densities:
   $$\log p(\theta \mid \mathcal{D}) = \log p(\theta) + \sum_{i=1}^N \log p(y_i \mid \theta) + \text{const}$$
   The acceptance ratio in log-space is simply the difference:
   $$\Delta \ell = \log p(\theta_{\text{prop}} \mid \mathcal{D}) - \log p(\theta_{\text{curr}} \mid \mathcal{D})$$
   $$\log \alpha = \min(0.0, \Delta \ell)$$
   The decision rule evaluates:
   $$\log(u) < \Delta \ell \quad \text{where } u \sim \text{Uniform}(0, 1)$$
   This formulation evaluates stably for $N=10^6$ observations without a single
   underflow error.

3. **Arbitrary Proposal Scale Pathology:**
   The AI picked an arbitrary `step_size = 1.0` without explaining that step
   size controls acceptance rate. If step size is too small ($0.05$), acceptance
   is $99\%$ but exploration is sluggish random walk diffusion; if too large
   ($20.0$), acceptance collapses to $0.5\%$.

## Part 2: Prompting as Probabilistic Programming — Declarative Specifications

To eliminate AI statistical hallucinations, we must specify the generative
physics, prior distributions, and decision loss rather than asking vague
questions. Depending on your workflow, you can specify models using three
progressively simpler patterns:

---

### Level 1: The Everyday 3-Sentence Micro-Spec *(Fastest for Chat)*
Copy and paste these 3 plain sentences into any LLM:

```markdown
1. Model: Non-conjugate posterior with N = 300 observations.
2. Numerical Invariant: Compute acceptance probability strictly in log-posterior
   space Delta_ell = ell(prop) - ell(curr) to prevent floating-point underflow.
3. Proposal Tuning: Tune proposal scale sigma_step to maintain an acceptance
   rate between 23% and 50%.
```

---

### Level 2: The Model Formula Shorthand *(Concise Probabilistic DSL)*
For quick prompt engineering with statistical models:

```text
Log-Accept: log(alpha) = min(0, log P(D|prop) + log P(prop) - log P(D|curr) - log P(curr))
Update:     Accept if log(u) < log(alpha), where u ~ Uniform(0, 1)
Tuning:     Adjust proposal scale to achieve target 44% acceptance
```

---

### Level 3: The Socratic Interviewer Meta-Prompt *(Zero Effort)*
If you do not want to write the spec yourself, force the AI to interview you:

> *"Act as an MCMC Compiler Architect. I am writing a Metropolis-Hastings sampler
for 300 observations. Ask me 3 questions about log-likelihood summation and
proposal scale tuning before writing the sampling loop."*

---

<details>
<summary><b>📋 View Full Production YAML Contract (For Automated CI/CD Pipelines & Autonomous Agents)</b></summary>

```yaml
# ==============================================================================
# PROBABILISTIC SPECIFICATION: METROPOLIS MCMC NUMERICAL STABILITY CONTRACT
# ==============================================================================
module: "ai/09_mcmc_mechanics_from_scratch"
target_parameter:
  name: "mu"
  domain: "(-infinity, +infinity)"

observables:
  sample_size: 300
  data_vector: "observed_heights_or_latencies"

prior_model:
  distribution: "Normal(loc=170.0, scale=15.0)"
  evaluation: "stats.norm.logpdf(mu, 170.0, 15.0)"

likelihood_model:
  distribution: "Normal(loc=mu, scale=8.0)"
  evaluation: "np.sum(stats.norm.logpdf(data, loc=mu, scale=8.0))"
  raw_probability_products_prohibited: true

computational_invariants:
  arithmetic_space: "log_posterior"
  acceptance_rule:
    log_delta: "log_joint(prop_mu) - log_joint(curr_mu)"
    decision_condition: "log(uniform(0, 1)) < log_delta"
  proposal_kernel:
    type: "Gaussian_random_walk"
    initial_step_size: 2.0
    target_acceptance_rate_1d: 0.44
    target_acceptance_window: "[0.25, 0.55]"
  warmup_contract:
    total_iterations: 6000
    warmup_discard_count: 1000
    retained_samples: 5000

decision_loss_contract:
  summary_metrics: ["mean", "std", "median", "hdi_95", "acceptance_rate"]
  numerical_health_assertion: "assert np.all(np.isfinite(samples))"
```

</details>


## Part 3: The 4 Mandatory Agentic Verification Gates

Any MCMC sampling engine developed or executed by an AI agent must pass four
mandatory verification gates:

### Gate 1: Underflow Invariant Gate (Zero Probability Products)
* **Invariant:** Under no circumstances may `np.prod` or raw probability density
  functions be multiplied across data points.
* **Verification Logic:**
  Static and runtime inspection must assert that all likelihood evaluations use
  `logpdf` combined with `np.sum()`. The output trace must satisfy:
  `assert np.all(np.isfinite(trace))`. If any `NaN` or `Inf` appears, the
  sampler is instantly invalidated.

### Gate 2: Log-Acceptance Bound Gate
* **Invariant:** The log-acceptance probability must satisfy:
  $$\log \alpha = \min(0.0, \Delta \ell) \le 0.0$$
* **Verification Logic:**
  Assert that when $\Delta \ell > 0$ (a uphill move to higher probability), the
  proposal is accepted with probability exactly $1.0$ ($\log \alpha = 0.0$).
  When $\Delta \ell < 0$, the decision evaluates $\log u < \Delta \ell$.

### Gate 3: Acceptance Rate Calibration Gate
* **Invariant:** For a 1D Gaussian random-walk Metropolis sampler, the
  production acceptance rate must fall within the optimal efficiency window:
  $$0.25 \le \text{Acceptance Rate} \le 0.55 \quad (\text{Optimal Target } \approx 0.44)$$
* **Verification Logic:**
  During the retained production run (post-warmup), calculate the empirical
  acceptance fraction. If acceptance $< 0.20$, the proposal step size is too
  large; if acceptance $> 0.60$, the step size is too small.

### Gate 4: Posterior Recovery Gate (Analytical Benchmark Verification)
* **Invariant:** On a 1D normal conjugate model:
  $$y \sim \mathcal{N}(\mu, \sigma_0^2), \quad \mu \sim \mathcal{N}(\mu_0, \tau_0^2)$$
  the empirical MCMC sample mean $\bar{\mu}_{\text{MCMC}}$ and standard
  deviation $s_{\mu}$ must recover exact analytical posterior parameters within
  $2$ Monte Carlo standard errors:
  $$| \bar{\mu}_{\text{MCMC}} - \mu_{\text{post}} | \le 2 \times \frac{s_{\mu}}{\sqrt{\text{ESS}}}$$

## Part 4: Worked Agent Execution Transcript & Telemetry

When an epistemic AI agent executes the calibrated Metropolis specification on
the 50-observation height dataset ($N=50, \bar{y} = 172.52, \sigma = 8.0$),
it produces the following verified execution transcript:

```text
[STEP 1: ANALYTICAL GROUND TRUTH BENCHMARK]
  Data Summary: N = 50, y_bar = 172.52, sigma_known = 8.0
  Prior: mu ~ Normal(mu_0 = 170.0, tau_0 = 15.0)
  Analytical Posterior Precision:
    tau_post^(-2) = (1 / 15^2) + 50 / (8^2) = 0.00444 + 0.78125 = 0.78569
    tau_post = 1 / sqrt(0.78569) = 1.1281 cm
  Analytical Posterior Mean:
    mu_post = tau_post^2 * (170 / 15^2 + 50 * 172.52 / 8^2)
    mu_post = 1.2727 * (0.7556 + 134.7812) = 172.505 cm

[STEP 2: RUNNING NAIVE LINEAR PRODUCT SAMPLER (COMPARISON)]
  Evaluating raw likelihood product:
    p_curr = np.prod(norm.pdf(heights, 170.0, 8.0)) = 1.48e-71
  For N = 300 data points:
    p_curr = np.prod(norm.pdf(data_300, 170.0, 8.0)) = 0.000000e+00 (UNDERFLOW!)
    Acceptance ratio: 0.0 / 0.0 = NaN. Execution aborted.

[STEP 3: RUNNING CALIBRATED LOG-SCALE METROPOLIS ENGINE]
  Total Draws: 6,000 (1,000 Warmup Discarded + 5,000 Retained)
  Proposal Step Size: sigma_step = 2.0
  Log-Posterior Function:
    log_joint(mu) = -0.5 * ((mu - 170) / 15)^2 - 0.5 * sum(((y - mu) / 8)^2)
  Initial Walker State: mu_0 = 150.0 (Far away from mode)
  
  Warmup Phase (Steps 0 - 1000):
    Burn-in drift from 150.0 to 172.4 completed in 65 steps.
    Warmup discarded completely.
  
  Production Phase (Steps 1000 - 6000):
    Total proposals: 5,000
    Accepted proposals: 2,184
    Empirical Acceptance: 2184 / 5000 = 43.68% (PASSED: Optimal 44%!)

[STEP 4: POSTERIOR CONVERGENCE & RECOVERY]
  Empirical Posterior Mean:  172.496 cm (Analytical Truth: 172.505 cm)
  Empirical Posterior StdDev:  1.134 cm (Analytical Truth:  1.128 cm)
  Monte Carlo Error: SE_MC = 1.134 / sqrt(650) = 0.044 cm
  Z-Score Error: |172.496 - 172.505| / 0.044 = 0.20 standard errors (PASSED Gate 4!)
  95% Credible Interval: [170.28, 174.72] cm
```

### ASCII MCMC Trace & Empirical Posterior Density
```
  Posterior Distribution: Analytical Gaussian Curve vs. MCMC Samples
 
  Density
    |
  0.35+                     * * * *  (Analytical Posterior: N(172.51, 1.13^2))
      |                   * : : : : *
  0.25+                 * : : : : : : *     [::: MCMC Sample Histogram]
      |                * : : : : : : : *
  0.15+               * : : : : : : : : *
      |              * : : : : : : : : : *
  0.05+             * : : : : : : : : : : *
      |            * : : : : : : : : : : : *
   0.0+-----------*-------------------------*-------------------------> mu (cm)
                169.0       171.0       173.0       175.0       177.0
                              ^           ^
                              |           +-- Mode: 172.50 cm
                              +-- 95% HDI: [170.28, 174.72]
```

### Decision Telemetry Table
| Parameter Setup | Step Size | Acceptance Rate | Effective Sample Size | MCMC Mean | Status |
|---|---|---|---|---|---|
| **Naive Linear Code ($N=300$)** | $1.0$ | **$0.0\%$ (Crash)** | $0$ (Frozen) | `NaN` | **Fatal Bug** |
| **Log-Scale (Tiny Step)** | $0.1$ | $96.8\%$ | $48$ (Sluggish) | $171.82$ | **Poor Mixing** |
| **Log-Scale (Huge Step)** | $25.0$ | $2.4\%$ | $32$ (Frozen) | $172.91$ | **Over-Stepped** |
| **Log-Scale (Calibrated)** | **$2.0$** | **$43.7\%$** | **$680$ (Optimal)** | **$172.50$** | **Certified Clean** |

## Part 5: Naive AI Prompt vs. Calibrated Epistemic Specification

| Dimension | Naive AI Prompt | Calibrated Epistemic Specification |
|---|---|---|
| **Likelihood Arithmetic** | Direct multiplication `np.prod(pdf)` (underflows to 0 for $N>150$) | Log-space summation `np.sum(logpdf)` (numerically stable for all $N$) |
| **Acceptance Ratio** | Computes ratio in linear space $P^*/P$, causing `0.0/0.0 = NaN` | Computes difference $\Delta \ell = \ell^* - \ell$, evaluating $\log u < \Delta \ell$ |
| **Proposal Calibration** | Uses unprincipled static step sizes; ignores efficiency | Targets optimal acceptance ($44\%$ for 1D, $23.4\%$ for multi-D) |
| **Warmup Discard** | Frequently pools initial transient steps into posterior | Strictly discards pre-convergence warmup draws ($1000$ steps) |
| **Verification Gates** | None; runs script and reports whatever numbers finish | 4 gates: Underflow check, Log-bounds, Acceptance calibration, Benchmark |
| **Production Risk** | Silent sampler freeze or `NaN` crashes in CI/CD | Certified stable, automated convergence monitoring |

## Part 6: 🎯 Check Your Understanding — 3-Tier "Audit the AI" Exercises

### 🟢 Tier 1: Spot the AI Hallucination
An engineer asks an AI model to troubleshoot a custom MCMC sampler:
> *"I ran my Metropolis sampler on $N=500$ server response times. The walker
> moves for the first 3 steps, then freezes completely at $\mu = 168.4$ for the
> remaining 9,997 steps. The AI responds: 'Your model has converged to the exact
> global maximum with 100% confidence. Because the posterior mode has been
> located, the acceptance probability for any other value is zero, so the
> sampler correctly remains at the exact summit.' "*

Explain why the AI's explanation is completely fabricated, and identify the true
computational failure.

<details>
<summary><b>Click here to view the solution...</b></summary>

**Forensic Audit:**
The AI has hallucinated that an MCMC sampler is an optimizer that stops moving
upon finding a mode!
1. **MCMC Never Freezes at Equilibrium:** A valid Markov chain never stops
   wandering; it must continue exploring the posterior distribution in
   accordance with its density.
2. **True Cause (Floating-Point Underflow):** The sampler used raw probability
   multiplication `np.prod`. On step 4, the current position achieved a
   likelihood small enough that `np.prod` underflowed to exact `0.0`. Then
   `p_curr = 0.0`. In subsequent steps, any proposed candidate also underflows
   to `p_prop = 0.0`, resulting in `alpha = 0.0 / 0.0 = NaN`. The comparison
   `uniform < NaN` evaluates to `False`, rejecting all proposals indefinitely.
</details>

---

### 🟡 Tier 2: Refactor the Flawed Specification
Below is an uncalibrated agent prompt to implement a Metropolis-Hastings
sampler with an asymmetric proposal distribution:
$$q(\theta^* \mid \theta) = \text{LogNormal}(\mu=\log \theta, \sigma=0.5)$$

```text
Write a Metropolis sampler for a positive scale parameter sigma > 0. At each
step, propose candidate sigma_prop ~ LogNormal(mean=log(sigma_curr), sigma=0.5).
Calculate the log-posterior difference delta_log = log_post(sigma_prop) -
log_post(sigma_curr) and accept if log(uniform(0, 1)) < delta_log.
```

Explain why this sampler violates detailed balance, and rewrite the prompt into
a calibrated declarative specification that includes the Hastings correction
term.

<details>
<summary><b>Click here to view the solution...</b></summary>

**Why it fails:**
Because the Log-Normal proposal is asymmetric, proposing $\theta^* \to \theta$
does not have the same probability density as proposing $\theta \to \theta^*$:
$$q(\theta^* \mid \theta) \neq q(\theta \mid \theta^*)$$
Omitting the Hastings correction factor:
$$\log q(\theta_{\text{curr}} \mid \theta_{\text{prop}}) - \log q(\theta_{\text{prop}} \mid \theta_{\text{curr}})$$
violates detailed balance, biasing samples toward larger values.

**Calibrated Declarative Specification:**
```yaml
module: "calibrated_metropolis_hastings_asymmetric"
target_parameter:
  name: "sigma"
  domain: "(0.0, +infinity)"

proposal_kernel:
  family: "LogNormal"
  distribution: "theta_prop ~ LogNormal(meanlog=log(theta_curr), sdlog=tau)"
  hastings_correction_log: "log(theta_prop) - log(theta_curr)"

acceptance_rule:
  log_alpha: "delta_log_joint + hastings_correction_log"
  decision: "log(uniform(0, 1)) < min(0.0, log_alpha)"

validation:
  assert_detailed_balance: "verify Hastings ratio cancels proposal asymmetry"
```
</details>

---

### 🔴 Tier 3: Architect an Autonomous Agent Contract
Architect an autonomous agent contract in pseudo-code that implements an
adaptive proposal tuner during warmup: using Robbins-Monro stochastic
approximation, it dynamically adjusts proposal scale $\sigma_{\text{step}}$ to
achieve target acceptance $0.44$, freezes adaptation strictly at step $W$, and
verifies that post-warmup acceptance stays within $[0.35, 0.55]$.

<details>
<summary><b>Click here to view the solution...</b></summary>

**Autonomous Contract Architecture:**
```python
def calibrated_adaptive_metropolis(
    log_joint_fn, init_theta, n_draws=6000, n_warmup=1000
):
  theta = float(init_theta)
  log_sigma_step = np.log(1.0)  # Initial proposal scale
  target_accept = 0.44
  samples = []
  accepted_post_warmup = 0

  curr_log_p = log_joint_fn(theta)

  for t in range(1, n_draws + 1):
    sigma_step = np.exp(log_sigma_step)
    prop_theta = theta + np.random.normal(0, sigma_step)
    prop_log_p = log_joint_fn(prop_theta)

    # Log-space Metropolis decision
    delta_log = prop_log_p - curr_log_p
    log_u = np.log(np.random.uniform(1e-12, 1.0))
    accepted = log_u < min(0.0, delta_log)

    if accepted:
      theta = prop_theta
      curr_log_p = prop_log_p

    # Robbins-Monro adaptation during warmup ONLY
    if t <= n_warmup:
      gamma_t = 1.0 / (t**0.7)  # Decreasing step size
      alpha_t = min(1.0, np.exp(delta_log)) if np.isfinite(delta_log) else 0.0
      log_sigma_step += gamma_t * (alpha_t - target_accept)
    else:
      # Production phase: Proposal scale FROZEN
      samples.append(theta)
      if accepted:
        accepted_post_warmup += 1

  # Post-sampling verification
  production_draws = n_draws - n_warmup
  emp_accept_rate = accepted_post_warmup / production_draws

  assert (
      0.30 <= emp_accept_rate <= 0.60
  ), f'Acceptance rate {emp_accept_rate:.3f} outside valid window'

  return {
      'samples': np.array(samples),
      'final_step_size': float(np.exp(log_sigma_step)),
      'production_acceptance_rate': float(emp_accept_rate),
      'warmup_discarded': n_warmup,
  }
```
</details>

## Key Takeaways & Course Navigation

### 🎯 Key Takeaways:
1. **Never Multiply Linear Probabilities:** Likelihood products underflow
   to zero for sample sizes as small as $N=150$. All Bayesian computation must
   occur on the log-posterior scale.
2. **Acceptance Difference Rule:** In log-space, the acceptance ratio is a
   simple subtraction: $\Delta \ell = \ell^* - \ell$. The acceptance decision
   evaluates $\log u < \Delta \ell$.
3. **The Acceptance Tradeoff:** Aim for $\approx 44\%$ acceptance in 1D models
   and $\approx 23.4\%$ in multi-dimensional models. Tiny steps cause sluggish
   random-walk diffusion; huge steps cause frequent rejections.
4. **Freeze Adaptation Before Production:** If you tune proposal step size, you
   must freeze it during the retained sampling phase. Adapting the kernel
   during production breaks the Markov property and destroys the stationary
   distribution.

---

**[🏠 Course Home](../README.md)** | **📖 [Conceptual](../conceptual/09_mcmc_mechanics_from_scratch.ipynb)** | **🤖 [AI-Native](09_mcmc_mechanics_from_scratch.ipynb)** | **🐍 [Python](../python/09_mcmc_mechanics_from_scratch.ipynb)** | **🔵 [R](../r/09_mcmc_mechanics_from_scratch.ipynb)** | **⬅️ [Prev: Ch 08](08_markov_chains_and_detailed_balance.ipynb)** | **➡️ [Next: Ch 10](10_mcmc_production_diagnostics.ipynb)**